In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("data_analysing")
    .config("spark.hadoop.fs.defaultFS", "file:///")   
    .getOrCreate()
)

spark


In [2]:
parquet_path = r"E:\BIGDATA\data\parquet\cic_ddos_2019_10gb_cleaned.parquet"

df_raw = spark.read.parquet(parquet_path)
df_raw.printSchema()

root
 |-- source_ip: string (nullable = true)
 |-- source_port: integer (nullable = true)
 |-- destination_ip: string (nullable = true)
 |-- destination_port: integer (nullable = true)
 |-- protocol: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- flow_duration: long (nullable = true)
 |-- total_fwd_packets: integer (nullable = true)
 |-- total_backward_packets: integer (nullable = true)
 |-- total_length_of_fwd_packets: double (nullable = true)
 |-- total_length_of_bwd_packets: double (nullable = true)
 |-- fwd_packet_length_max: double (nullable = true)
 |-- fwd_packet_length_min: double (nullable = true)
 |-- fwd_packet_length_mean: double (nullable = true)
 |-- fwd_packet_length_std: double (nullable = true)
 |-- bwd_packet_length_max: double (nullable = true)
 |-- bwd_packet_length_min: double (nullable = true)
 |-- bwd_packet_length_mean: double (nullable = true)
 |-- bwd_packet_length_std: double (nullable = true)
 |-- flow_bytes_per_s: double (nullabl

In [3]:
df = df_raw

In [4]:
total_rows = df.count()
print("Total rows (cleaned):", total_rows)

df.show(5, truncate=False)


Total rows (cleaned): 23968592
+----------+-----------+--------------+----------------+--------+--------------------------+-------------+-----------------+----------------------+---------------------------+---------------------------+---------------------+---------------------+----------------------+---------------------+---------------------+---------------------+----------------------+---------------------+-------------------+------------------+------------------+------------------+------------+------------+-------------+------------------+------------------+-----------+-----------+-------------+------------+-----------+-----------+-----------+-------------+-------------+-------------+-------------+-----------------+-----------------+------------------+-----------------+-----------------+-----------------+------------------+-----------------+----------------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+-

In [5]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

label_counts = (
    df.groupBy("label")
      .count()
      .orderBy(F.desc("count"))
)

label_counts.show(50, truncate=False)


+-------------+-------+
|label        |count  |
+-------------+-------+
|DrDoS_SNMP   |5159863|
|DrDoS_DNS    |5071002|
|DrDoS_MSSQL  |4522489|
|DrDoS_NetBIOS|4093273|
|DrDoS_LDAP   |2179928|
|Syn          |1380015|
|DrDoS_NTP    |1202639|
|UDP-lag      |330329 |
|BENIGN       |28615  |
|WebDDoS      |439    |
+-------------+-------+



In [6]:
label_ratio = (
    label_counts
    .withColumn("ratio", F.col("count") / F.lit(total_rows))
)

label_ratio.show(50, truncate=False)


+-------------+-------+---------------------+
|label        |count  |ratio                |
+-------------+-------+---------------------+
|DrDoS_SNMP   |5159863|0.2152760162132177   |
|DrDoS_DNS    |5071002|0.21156862280437666  |
|DrDoS_MSSQL  |4522489|0.18868396608361476  |
|DrDoS_NetBIOS|4093273|0.17077653122052391  |
|DrDoS_LDAP   |2179928|0.09094935572352351  |
|Syn          |1380015|0.05757597275634714  |
|DrDoS_NTP    |1202639|0.0501756214966653   |
|UDP-lag      |330329 |0.013781744042370115 |
|BENIGN       |28615  |0.00119385402363226  |
|WebDDoS      |439    |1.8315635728623525E-5|
+-------------+-------+---------------------+



In [7]:
numeric_types = (
    T.ByteType, T.ShortType, T.IntegerType, T.LongType,
    T.FloatType, T.DoubleType, T.DecimalType
)

numeric_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, numeric_types) and f.name != "unnamed_0"
]

print("Numeric cols:", len(numeric_cols))
print(numeric_cols)


Numeric cols: 81
['source_port', 'destination_port', 'protocol', 'flow_duration', 'total_fwd_packets', 'total_backward_packets', 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std', 'bwd_packet_length_max', 'bwd_packet_length_min', 'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytes_per_s', 'flow_packets_per_s', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length', 'bwd_header_length', 'fwd_packets_per_s', 'bwd_packets_per_s', 'min_packet_length', 'max_packet_length', 'packet_length_mean', 'packet_length_std', 'packet_length_variance', 'fin_flag_count', 'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag

In [ ]:
stats = []

for c in numeric_cols:
    row = df.select(
        F.lit(c).alias("column"),
        F.count(F.col(c)).alias("count"),
        F.mean(F.col(c)).alias("mean"),
        F.stddev(F.col(c)).alias("std"),
        F.min(F.col(c)).alias("min"),
        F.max(F.col(c)).alias("max")
    )
    stats.append(row)

from functools import reduce
stats_df = reduce(lambda a, b: a.unionByName(b), stats)
stats_df.show(truncate=False)


+-------+------------------+------------------+------------------+------------------+-----------------+----------------------+---------------------------+---------------------------+---------------------+---------------------+----------------------+---------------------+---------------------+---------------------+----------------------+---------------------+----------------+--------------------+------------------+-------------------+------------------+------------------+-----------------+----------------+-------------------+------------------+------------------+------------------+------------------+-------------------+-----------------+-------------------+--------------------+-------------+-------------+-------------+---------------------+------------------+-----------------+------------------+-----------------+-----------------+------------------+------------------+----------------------+--------------+--------------------+--------------------+--------------+-------------------+------

In [10]:
df_bin = df.withColumn(
    "label_bin",
    F.when(F.col("label") == "BENIGN", F.lit(1)).otherwise(F.lit(0))
)

df_bin.groupBy("label_bin").count().orderBy("label_bin").show()


+---------+--------+
|label_bin|   count|
+---------+--------+
|        0|23939977|
|        1|   28615|
+---------+--------+



In [11]:
use_cols = [c for c in numeric_cols if c not in ["source_port", "destination_port"]]

agg_exprs = []
for c in use_cols:
    agg_exprs.append(F.mean(F.col(c)).alias(f"{c}_mean"))
    agg_exprs.append(F.stddev(F.col(c)).alias(f"{c}_std"))

stats_bin = (
    df_bin.groupBy("label_bin")
          .agg(*agg_exprs)
)

stats_bin.show(truncate=False)


+---------+------------------+------------------+------------------+------------------+----------------------+---------------------+---------------------------+--------------------------+--------------------------------+-------------------------------+--------------------------------+-------------------------------+--------------------------+-------------------------+--------------------------+-------------------------+---------------------------+--------------------------+--------------------------+-------------------------+--------------------------+-------------------------+--------------------------+-------------------------+---------------------------+--------------------------+--------------------------+-------------------------+---------------------+--------------------+-----------------------+----------------------+------------------+-----------------+-----------------+-----------------+------------------+-------------------+------------------+------------------+---------------

In [12]:
spark.stop()